# 05 · Challenger models (H4, H5)

**Goal:** train night-light, embedding and small-area-estimation models on a simulated household survey and
score them on the localities the survey never visited, for Indigenous and non-Indigenous places separately.
**Time:** 10-20 minutes (20 survey replications). **Needs:** notebook 04 finished.

In [ ]:
# Make the project importable; works in JupyterLab, VS Code and Colab without reinstalling.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from mapaudit import notebook as nb
from mapaudit import pipeline, report
from mapaudit.config import get_paths

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
MODE = nb.notebook_mode()  # "real" for you; CI sets MAPAUDIT_NOTEBOOK_MODE=demo to test this notebook
print("Mode:", MODE)

In [ ]:
if MODE == "demo":
    nb.prepare_demo(through="build")
info = pipeline.simulate(MODE)
report.report(MODE)
info

## How the models are scored

Fixed in the pre-analysis plan before any real data were seen:

- **RMSE** of the predicted deprivation rate is the accuracy measure, because the budget formula adds up
  predicted rates and squared error is the loss whose best prediction is the mean.
- **Signed bias** is the fairness measure: does a model systematically understate deprivation for a group?
- **Calibration slope** (above 1 = predictions squeezed toward the average) and **variance ratio** (below 1 =
  the same) show whether a low RMSE was bought by compressing the extremes, where programs act.
- Everything is computed **per person** (population-weighted, primary) and **per community** (unweighted).
- No MAPE or MSLE: deprivation rates are often zero, and both measures discount errors in the poorest places.

## H4 · Which model maps deprivation best, and for whom?

In [ ]:
nb.show_figure("challenger_bias", MODE)
metrics = nb.load_table("challenger_metrics_summary", MODE)
people = metrics[metrics["weighting"] == "people"]
people.pivot(index="model", columns="group", values=["bias_mean", "rmse_mean", "calibration_slope_mean"]).round(3)

H4 as pre-registered: the difference in absolute bias for Indigenous localities between gradient boosting and
small area estimation, computed on the same simulated survey in each replication, so the luck of the draw
cancels out. Supported if the 95% replication range lies above zero. The other rows compare each model's
RMSE with small area estimation the same way.

In [ ]:
paired = nb.load_table("challenger_paired", MODE)
paired["hypothesis"] = paired["hypothesis"].fillna("")
paired["supported"] = paired["supported"].fillna("")
paired.round(4)

## Where the errors fall: bias by true deprivation decile

Compression shows up as positive bias in the least deprived deciles and negative bias in the most deprived
ones. Per-community metrics weight each locality equally, which matters because Indigenous localities are
many and small.

In [ ]:
nb.show_figure("challenger_deciles", MODE)
places = metrics[metrics["weighting"] == "places"]
places.pivot(index="model", columns="group", values=["bias_mean", "rmse_mean"]).round(3)

## H5 · Are the 90% intervals honest for everyone?

"Survey frame" = localities large enough for the simulated survey to visit. Conformal guarantees only hold
where calibration and prediction look alike. The interval score adds width and a penalty for every miss, so
it rewards intervals that are both narrow and honest (lower is better).

In [ ]:
nb.show_figure("conformal_coverage", MODE)
coverage = nb.load_table("challenger_coverage_summary", MODE)
coverage[coverage["group"] != "All"].pivot_table(
    index=["model", "method"], columns=["frame", "group"], values=["coverage_mean", "interval_score_mean"]
).round(3)

## Targeting and money

In [ ]:
targeting = nb.load_table("challenger_targeting_summary", MODE)
display(targeting.pivot(index="model", columns="group", values="exclusion_rate_mean").round(3))
nb.show_figure("allocation", MODE)
nb.load_table("challenger_municipal_summary", MODE).round(3)

Who these comparisons cover (localities with suppressed census data cannot be scored):

In [ ]:
nb.load_table("challenger_evaluation_coverage", MODE).round(3)

## Your notes (for lab-notebook posts 4 and 5)

- Which model wins overall? Does borrowing strength (EB, Fay-Herriot) shrink the Indigenous bias gap?
- Is H4 supported? Quote the paired difference and its replication range.
- Does any model buy a low RMSE by compressing the extremes (calibration slope well above 1)?
- Coverage inside vs. outside the survey frame: what does that imply for survey design?

## Save your work to GitHub

1. Run the next cell (real data only). It runs the checks GitHub will run: formatting, lint, tests and the
   website build. Every row should say `pass`.
2. Commit and push what changed, with a message that says what you did. Nothing from `data/` goes to
   GitHub, and no notebook output may show locality-level rows.
3. On GitHub, the **Actions** tab reruns the checks; every run should go green within about 10 minutes.

In [ ]:
if MODE == "real":
    display(nb.run_checks())